# Underfitting and overfitting: interactive

Run the cell with **Shift + Enter**, then move the slider.

The slider sets the model's **complexity**: how many bends its curve is allowed to have.
For each setting, the title shows two errors (lower is better):

- **training error:** how well the curve fits the 12 points it learned from,
- **new-data error:** how well it predicts points it has never seen.

**Try:** find the complexity with the lowest new-data error. Notice that the training error keeps falling all the way to 0,
while the new-data error is lowest around complexity 5-6 and then rises. That rise is overfitting. (With only 12 points the rise is a little bumpy: complexity 10 happens to dip, but the trend is clear.)

In [ ]:
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(31)
truth = lambda x: np.sin(2 * np.pi * x)                       # the real pattern (unknown to the model)
x_train = np.sort(rng.uniform(0, 1, 12))
y_train = truth(x_train) + rng.normal(0, 0.25, 12)            # 12 noisy training points
lo, hi = x_train.min(), x_train.max()
x_new = rng.uniform(lo, hi, 300)                              # new data, never used for training
y_new = truth(x_new) + rng.normal(0, 0.25, 300)
grid = np.linspace(lo, hi, 400)

def rmse(coefs, x, y):
    return np.sqrt(np.mean((np.polyval(coefs, x) - y) ** 2))  # typical size of the error

degrees = range(1, 12)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_train, y=y_train, mode="markers", name="training points", marker=dict(size=10, color="#4C78A8")))
titles = []
for d in degrees:
    coefs = np.polyfit(x_train, y_train, d)                   # fit a curve with d bends' worth of flexibility
    fig.add_trace(go.Scatter(x=grid, y=np.polyval(coefs, grid), mode="lines", visible=(d == 3),
                             line=dict(width=3, color="#F58518"), name=f"complexity {d}"))
    titles.append(f"Complexity {d}: training error {rmse(coefs, x_train, y_train):.2f}, "
                  f"new-data error {rmse(coefs, x_new, y_new):.2f}")

steps = [dict(method="update", label=str(d),
              args=[{"visible": [True] + [j == i for j in range(len(degrees))]}, {"title.text": titles[i]}])
         for i, d in enumerate(degrees)]
fig.update_layout(title=titles[2], sliders=[dict(active=2, steps=steps, currentvalue=dict(prefix="Complexity: "))],
                  yaxis_range=[-2, 2], xaxis_title="Input", yaxis_title="Output", height=520, template="simple_white")
fig.show()